# Phase 6: CI/CD Pipeline — SageMaker Pipelines DAG
**AAI-540 MLOps | CX Cortalyst — Customer Sentiment Analysis & GenAI Response Automation**  
**Author:** Jagadeesh Kumar Sellappan

## Overview
This notebook implements the full CI/CD pipeline as a **SageMaker Pipelines DAG** — wrapping all existing code from Phase 4 into an automated, reproducible, version-controlled pipeline.

### 5-Step Pipeline DAG:
1. **DataPreprocessing** (Processing Job) — Read splits from S3, prepare XGBoost CSVs
2. **XGBoostTraining** (Training Job) — XGBoost classifier training
3. **ModelEvaluation** (Processing Job) — Compute F1, Accuracy, AUC-ROC
4. **F1QualityGate** (Condition Step) — Quality gate: F1 >= threshold?
   - **PASS →** RegisterModel (register approved model in the Model Registry)
   - **FAIL →** QualityGateFailed (halt; production model preserved)
5. **BatchTransform** (Transform Job) — Classify production reviews (runs only after RegisterModel)

### Downstream (separate module — not a DAG step):
- BatchTransform writes raw **CSV** scores to `pipeline/predictions/` (native SageMaker XGBoost output). Section **5b** curates these into `data/predictions/production_predictions.parquet` (adds `review_id` / `predicted_label`), so the CI/CD pipeline itself produces the GenAI input.
- **GenAI Response Automation** runs as a **decoupled module** *after* the pipeline. In production it is triggered by an **S3 event notification** when that curated parquet lands, and generates draft emails for Negative reviews. The runnable demo path produces the same parquet from Phase 4 (`04_model.ipynb`). See Phase 7 (07_GenAI_Response.ipynb).

### Demonstrates:
- ✅ **Success state** — all 5 steps complete, model registered and batch predictions written
- ❌ **Failure state** — Step 4 condition fails, pipeline halts, production model preserved

**NOTE:** Raw data preprocessing (JSON flattening, embedding generation,
and 5% stratified sampling) was performed in Phase 3 notebook
(03_Feature_Engineering_and_Splitting.ipynb) and stored as parquet
splits in S3. In a production environment, this preprocessing step
would be included as Step 0 of the pipeline, triggered by new raw
data arriving in S3.

For this pipeline, we read from pre-processed splits to focus
on the MLOps orchestration components (training, evaluation,
quality gates, registry, deployment) which are the core
deliverables of this course project.

---
## 1. Environment Setup

In [1]:
!pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops sagemaker-studio 

Found existing installation: sagemaker 2.232.2


Uninstalling sagemaker-2.232.2:


  Successfully uninstalled sagemaker-2.232.2


Found existing installation: sagemaker-core 1.0.78
Uninstalling sagemaker-core-1.0.78:
  Successfully uninstalled sagemaker-core-1.0.78


In [2]:
!pip install -q sagemaker==2.232.2 --no-deps

In [4]:
!pip install -q sagemaker-core==1.0.78

In [5]:
!pip install -q boto3 awswrangler pandas scikit-learn sentence-transformers

In [13]:
import boto3
import sagemaker
import json
import time
import os
import pandas as pd
import numpy as np
import awswrangler as wr
import warnings
warnings.filterwarnings('ignore')

from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, TransformStep
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.inputs import TrainingInput, TransformInput
from sagemaker.transformer import Transformer
from sagemaker.model import Model
from sagemaker.pytorch import PyTorchProcessor

# ── Configuration ──────────────────────────────────────────────
REGION         = 'us-east-1'
BUCKET         = 'aai-540-group8-yelp-data-301798465569-us-east-1-an'
MODEL_DATA     = 's3://aai-540-group8-yelp-data-301798465569-us-east-1-an/models/xgboost-sentiment/sagemaker-xgboost-2026-06-14-11-45-03-373/output/model.tar.gz'
MODEL_GROUP    = 'cx-cortalyst-sentiment-models'
PIPELINE_NAME  = 'cx-cortalyst-sentiment-pipeline'

S3_MODELS      = f's3://{BUCKET}/models/pipeline/'
S3_EVAL        = f's3://{BUCKET}/pipeline/evaluation/'
S3_PREDICTIONS = f's3://{BUCKET}/pipeline/predictions/'

# ── Sessions ───────────────────────────────────────────────────
boto_session      = boto3.Session(region_name=REGION)
sagemaker_session = sagemaker.Session(boto_session=boto_session)
sm_client         = boto3.client('sagemaker',  region_name=REGION)
s3_client         = boto3.client('s3',         region_name=REGION)
role              = sagemaker.get_execution_role(sagemaker_session=sagemaker_session)
container         = sagemaker.image_uris.retrieve('xgboost', REGION, '1.7-1')

# ── PipelineSession ────────────────────────────────────────────
from sagemaker.workflow.pipeline_context import PipelineSession
pipeline_session = PipelineSession(
    boto_session=boto_session,
    default_bucket=BUCKET
)

print('✅ Environment initialized')
print(f'   Region:   {REGION}')
print(f'   Bucket:   {BUCKET}')
print(f'   Pipeline: {PIPELINE_NAME}')

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


✅ Environment initialized
   Region:   us-east-1
   Bucket:   aai-540-group8-yelp-data-301798465569-us-east-1-an
   Pipeline: cx-cortalyst-sentiment-pipeline


---
## 2. Upload Pipeline Scripts to S3

SageMaker Processing Jobs need scripts stored in S3. We write them locally and upload.

In [14]:
# ──  preprocessing.py ─────────────────────────────
preprocessing_script = '''
import pandas as pd
import numpy as np
import os
import sys
import subprocess
import time

print("=" * 60)
print("CX CORTALYST — Pipeline Preprocessing Step")
print("=" * 60)

# ── Step 0: Install dependencies (Python 3.10 container) ──────
print("\\nStep 0: Installing dependencies...")
start = time.time()

# subprocess.check_call([
#     sys.executable, "-m", "pip", "install",
#     "sentence-transformers==5.1.2",  # ← works on Python 3.10
#     "safetensors==0.7.0",
#     "--quiet",
#     "--no-cache-dir"
# ])

# Force install older, stable versions compatible with the container's built-in PyTorch 2.0.1
subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers==2.2.2", "transformers==4.30.2"])

print(f"✅ Dependencies installed ({time.time()-start:.0f}s)")

from sentence_transformers import SentenceTransformer
print("✅ SentenceTransformer imported successfully")

# ── Step 1: Load data ─────────────────────────────────────────
print("\\nStep 1: Loading parquet splits from S3...")

df_train = pd.read_parquet("/opt/ml/processing/input/train/")
df_val   = pd.read_parquet("/opt/ml/processing/input/validation/")
df_test  = pd.read_parquet("/opt/ml/processing/input/test/")

print(f"✅ Loaded — Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")
print(f"   Columns: {df_train.columns.tolist()}")

# ── Step 2: Validate columns ──────────────────────────────────
print("\\nStep 2: Validating columns...")
for col in ["text", "sentiment_label"]:
    if col not in df_train.columns:
        raise ValueError(f"Required column {col} not found!")
    print(f"   OK: {col}")

neg = (df_train["sentiment_label"]==0).sum()
pos = (df_train["sentiment_label"]==1).sum()
print(f"   Class balance — Neg: {neg:,} | Pos: {pos:,}")

# ── Step 3: Stratified sample ─────────────────────────────────
print("\\nStep 3: Stratified sampling (2k per class = 4k total)...")

TARGET_PER_CLASS = 2000

def stratified_sample(df, n_per_class, seed=42):
    neg_avail = (df["sentiment_label"]==0).sum()
    pos_avail = (df["sentiment_label"]==1).sum()
    actual_n  = min(n_per_class, neg_avail, pos_avail)
    sampled = df.groupby("sentiment_label", group_keys=False)\
        .apply(lambda x: x.sample(min(actual_n, len(x)), random_state=seed))\
        .reset_index(drop=True)
    print(f"   {len(sampled):,} rows | Neg: {(sampled['sentiment_label']==0).sum():,} | Pos: {(sampled['sentiment_label']==1).sum():,}")
    return sampled

print("   Train:"); df_train = stratified_sample(df_train, TARGET_PER_CLASS)
print("   Val:");   df_val   = stratified_sample(df_val,   TARGET_PER_CLASS // 4)
print("   Test:");  df_test  = stratified_sample(df_test,  TARGET_PER_CLASS // 4)

print(f"✅ Sampling complete")
print(f"   Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")

# ── Step 4: Generate embeddings ───────────────────────────────
print("\\nStep 4: Loading embedding model (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Model loaded")

def embed(df, name):
    print(f"   Embedding {name} ({len(df):,} records)...")
    start = time.time()
    emb = embedder.encode(
        df["text"].fillna("").astype(str).tolist(),
        batch_size=16,
        show_progress_bar=False,
        convert_to_numpy=True
    )
    print(f"   ✅ {name}: {emb.shape} ({time.time()-start:.0f}s)")
    return pd.DataFrame(emb, columns=[f"emb_{i}" for i in range(emb.shape[1])])

train_emb = embed(df_train, "Train")
val_emb   = embed(df_val,   "Val")
test_emb  = embed(df_test,  "Test")

# ── Step 5: Build feature matrix ─────────────────────────────
print("\\nStep 5: Building feature matrix...")

STRUCTURED = ["text_char_length","is_elite","review_useful_votes",
              "review_funny_votes","review_cool_votes"]

def build(df, emb, name):
    available = [c for c in STRUCTURED if c in df.columns]
    y   = df["sentiment_label"].astype(int).reset_index(drop=True)
    X_s = df[available].fillna(0).reset_index(drop=True)
    X_e = emb.reset_index(drop=True)
    out = pd.concat([y, X_s, X_e], axis=1)
    print(f"   ✅ {name}: {out.shape} (1 label + {len(available)} structured + {X_e.shape[1]} embeddings)")
    return out

train_xgb = build(df_train, train_emb, "Train")
val_xgb   = build(df_val,   val_emb,   "Val")
test_xgb  = build(df_test,  test_emb,  "Test")

# ── Step 6: Write outputs ─────────────────────────────────────
print("\\nStep 6: Writing CSVs...")

os.makedirs("/opt/ml/processing/output/train",      exist_ok=True)
os.makedirs("/opt/ml/processing/output/validation", exist_ok=True)
os.makedirs("/opt/ml/processing/output/test",       exist_ok=True)

train_xgb.to_csv("/opt/ml/processing/output/train/train.csv",
                 header=False, index=False)
val_xgb.to_csv(  "/opt/ml/processing/output/validation/validation.csv",
                 header=False, index=False)
test_xgb.to_csv( "/opt/ml/processing/output/test/test.csv",
                 header=False, index=False)

for path, name in [
    ("/opt/ml/processing/output/train/train.csv",           "Train"),
    ("/opt/ml/processing/output/validation/validation.csv", "Val"),
    ("/opt/ml/processing/output/test/test.csv",             "Test")
]:
    size = os.path.getsize(path) / 1024 / 1024
    print(f"   ✅ {name}: {size:.1f} MB")

print("\\n" + "=" * 60)
print("PREPROCESSING COMPLETE")
print(f"  Train: {len(train_xgb):,} x {train_xgb.shape[1]} cols")
print(f"  Val:   {len(val_xgb):,} x {val_xgb.shape[1]} cols")
print(f"  Test:  {len(test_xgb):,} x {test_xgb.shape[1]} cols")
print(f"  Features: 1 label + 5 structured + 384 embeddings = 390 total")
print("=" * 60)
'''

In [15]:
# ── evaluate.py ───────────────────────────────────────────────
evaluate_script = '''
import pandas as pd
import numpy as np
import json
import os
import tarfile
import xgboost as xgb
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

print("Step 3: Evaluating model...")

# Load model
model_path = "/opt/ml/processing/model/model.tar.gz"
with tarfile.open(model_path) as tar:
    tar.extractall("/opt/ml/processing/model/")

booster = xgb.Booster()
booster.load_model("/opt/ml/processing/model/xgboost-model")
print("Model loaded")

# Load test data
df_test = pd.read_csv("/opt/ml/processing/input/test/test.csv", header=None)
y_test  = df_test.iloc[:, 0].values
X_test  = df_test.iloc[:, 1:].values

# Predict
dtest       = xgb.DMatrix(X_test)
y_pred_prob = booster.predict(dtest)
y_pred      = (y_pred_prob >= 0.5).astype(int)

# Compute metrics
acc = float(accuracy_score(y_test, y_pred))
f1  = float(f1_score(y_test, y_pred, average="macro"))
auc = float(roc_auc_score(y_test, y_pred_prob))

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {f1:.4f}")
print(f"AUC-ROC:  {auc:.4f}")

# Write evaluation report — SageMaker Pipelines reads this
report = {
    "binary_classification_metrics": {
        "accuracy": {"value": acc},
        "f1_macro": {"value": f1},
        "auc_roc":  {"value": auc}
    }
}

os.makedirs("/opt/ml/processing/evaluation", exist_ok=True)
with open("/opt/ml/processing/evaluation/evaluation.json", "w") as f:
    json.dump(report, f, indent=2)

print("Evaluation report written")
'''

In [16]:
# Write scripts locally
os.makedirs('pipeline_scripts', exist_ok=True)
with open('pipeline_scripts/preprocessing.py', 'w') as f:
    f.write(preprocessing_script)
with open('pipeline_scripts/evaluate.py', 'w') as f:
    f.write(evaluate_script)

# Upload to S3
for script in ['preprocessing.py', 'evaluate.py']:
    s3_client.upload_file(
        f'pipeline_scripts/{script}',
        BUCKET,
        f'pipeline/scripts/{script}'
    )
    print(f'Uploaded: pipeline/scripts/{script}')

print(f'\nAll scripts uploaded to s3://{BUCKET}/pipeline/scripts/')

Uploaded: pipeline/scripts/preprocessing.py
Uploaded: pipeline/scripts/evaluate.py

All scripts uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/pipeline/scripts/


---
## 3. Define Pipeline Steps

Each step wraps the existing code from Phase 4 into a SageMaker Pipelines step object.

In [17]:
print("Defining pipeline parameters and processors...")
from sagemaker.workflow.pipeline_context import PipelineSession

pipeline_session = PipelineSession(
    boto_session=boto_session,
    default_bucket=BUCKET
)
# ── F1 threshold parameter ─────────────────────────────────────
F1_THRESHOLD = 0.85

from sagemaker.processing import FrameworkProcessor
from sagemaker.pytorch import PyTorch

pytorch_processor = FrameworkProcessor(
    estimator_cls=PyTorch,
    framework_version='2.0',
    py_version='py310',
    role=role,
    instance_type='ml.m5.xlarge',
    instance_count=1,
    sagemaker_session=pipeline_session
)

# FrameworkProcessor uses run() with source_dir + entry_point
# This handles python execution correctly
step_args = pytorch_processor.run(
    code='preprocessing.py',
    source_dir='pipeline_scripts',   # local dir with preprocessing.py
    inputs=[
        ProcessingInput(source=f's3://{BUCKET}/data/splits/train/',      destination='/opt/ml/processing/input/train'),
        ProcessingInput(source=f's3://{BUCKET}/data/splits/validation/', destination='/opt/ml/processing/input/validation'),
        ProcessingInput(source=f's3://{BUCKET}/data/splits/test/',       destination='/opt/ml/processing/input/test')
    ],
    outputs=[
        ProcessingOutput(output_name='train',      source='/opt/ml/processing/output/train',      destination=f's3://{BUCKET}/pipeline/processed/train/'),
        ProcessingOutput(output_name='validation', source='/opt/ml/processing/output/validation', destination=f's3://{BUCKET}/pipeline/processed/validation/'),
        ProcessingOutput(output_name='test',       source='/opt/ml/processing/output/test',       destination=f's3://{BUCKET}/pipeline/processed/test/')
    ]
)

step_process = ProcessingStep(
    name='DataPreprocessing',
    step_args=step_args
)
print('✅ step_process defined with FrameworkProcessor')

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.


Defining pipeline parameters and processors...
✅ step_process defined with FrameworkProcessor


In [18]:
# ── Step 2: Training Job ───────────────────────────────────────
from sagemaker.estimator import Estimator

xgb_estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=S3_MODELS,
    sagemaker_session=sagemaker_session
)

xgb_estimator.set_hyperparameters(
    objective='binary:logistic',
    eval_metric='logloss',
    num_round=100,
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    scale_pos_weight=1.0
)

step_train = TrainingStep(
    name='XGBoostTraining',
    estimator=xgb_estimator,
    inputs={
        'train': TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs['train'].S3Output.S3Uri,
            content_type='text/csv'
        ),
        'validation': TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs['validation'].S3Output.S3Uri,
            content_type='text/csv'
        )
    }
)
print('Step 2 defined: XGBoostTraining')

Step 2 defined: XGBoostTraining


In [19]:
# ── Step 3: Evaluation Job ─────────────────────────────────────
from sagemaker.processing import ScriptProcessor

eval_processor = ScriptProcessor(
    image_uri=container,
    role=role,
    instance_type='ml.m5.xlarge',
    instance_count=1,
    command=['python3'],
    sagemaker_session=sagemaker_session
)

# PropertyFile reads evaluation.json output for the Condition Step
evaluation_report = PropertyFile(
    name='EvaluationReport',
    output_name='evaluation',
    path='evaluation.json'
)

step_evaluate = ProcessingStep(
    name='ModelEvaluation',
    processor=eval_processor,
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination='/opt/ml/processing/model'
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs['test'].S3Output.S3Uri,
            destination='/opt/ml/processing/input/test'
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name='evaluation',
            source='/opt/ml/processing/evaluation',
            destination=S3_EVAL
        )
    ],
    code=f's3://{BUCKET}/pipeline/scripts/evaluate.py',
    property_files=[evaluation_report]
)
print('Step 3 defined: ModelEvaluation')

Step 3 defined: ModelEvaluation


In [20]:
from sagemaker.model import Model
from sagemaker.model_metrics import ModelMetrics, MetricsSource
from sagemaker.workflow.model_step import ModelStep

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=f'{S3_EVAL}evaluation.json',
        content_type='application/json'
    )
)

# Create model with PipelineSession
xgb_model = Model(
    image_uri=container,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session   
)

# Register model
register_args = xgb_model.register(
    content_types=['text/csv'],
    response_types=['text/csv'],
    inference_instances=['ml.m5.xlarge'],
    transform_instances=['ml.m5.xlarge'],
    model_package_group_name=MODEL_GROUP,
    approval_status='Approved',
    model_metrics=model_metrics,
    description=f'XGBoost Sentiment Classifier — F1 threshold: {F1_THRESHOLD}'
)

step_register = ModelStep(
    name='RegisterModel',
    step_args=register_args
)

# ── Fail Step ──────────────────────────────────────────────────
step_fail = FailStep(
    name='QualityGateFailed',
    error_message=f'Model F1 below threshold {F1_THRESHOLD}. Production model preserved.'
)

cond_f1 = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_evaluate.name,
        property_file=evaluation_report,
        json_path='binary_classification_metrics.f1_macro.value'
    ),
    right=F1_THRESHOLD
)

# ── Condition Step ─────────────────────────────────────────────
step_condition = ConditionStep(
    name='F1QualityGate',
    conditions=[cond_f1],
    if_steps=[step_register],
    else_steps=[step_fail]
)

print(f'Step 4 defined: F1QualityGate (threshold: {F1_THRESHOLD})')
print(f'PASS → RegisterModel')
print(f'FAIL → QualityGateFailed')
print(f'Step 5 defined: RegisterModel (PipelineSession)')

Step 4 defined: F1QualityGate (threshold: 0.85)
PASS → RegisterModel
FAIL → QualityGateFailed
Step 5 defined: RegisterModel (PipelineSession)


In [21]:
# ── Step 6: Batch Transform ────────────────────────────────────
from sagemaker.workflow.steps import TransformStep, CreateModelStep
from sagemaker.inputs import TransformInput, CreateModelInput
from sagemaker.transformer import Transformer
from sagemaker.model import Model

# 1. Create the Model object
transform_model = Model(
    image_uri=container,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session
)

# 2. The Pipeline Step to actually build the model entity
step_create_model = CreateModelStep(
    name="CXCortalystCreateModel",
    model=transform_model,
    inputs=CreateModelInput(instance_type="ml.m5.xlarge")
)

# 3. Define the Transformer using the dynamic name from Step 2
transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type='ml.m5.xlarge',
    instance_count=1,
    output_path=S3_PREDICTIONS,
    accept='text/csv',
    assemble_with='Line',
    sagemaker_session=pipeline_session
)

# 4. Define the Transform Step
step_transform = TransformStep(
    name='BatchTransform',
    transformer=transformer,
    inputs=TransformInput(
        data=f's3://{BUCKET}/data/sagemaker_input/production/',
        content_type='text/csv',
        split_type='Line'
    )
)

print('Step 6 defined: BatchTransform')
print(f'Input:  s3://{BUCKET}/data/sagemaker_input/production/')
print(f'Output: {S3_PREDICTIONS}')

Step 6 defined: BatchTransform
Input:  s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/data/sagemaker_input/production/
Output: s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/pipeline/predictions/


---
## 4. Create & Run the Pipeline

In [22]:
# Recreate pipeline with explicit dependencies
print('Recreating pipeline with correct step dependencies...')

# Add explicit dependency — BatchTransform only runs AFTER RegisterModel
step_transform.add_depends_on([step_register])

pipeline = Pipeline(
    name=PIPELINE_NAME,
    steps=[
        step_process,
        step_train,
        step_evaluate,
        step_condition,
        step_transform
    ],
    sagemaker_session=pipeline_session
)

pipeline.upsert(role_arn=role)
print(f'Pipeline updated with correct dependencies')
print(f'DataPreprocessing → Training → Evaluation')
print(f'   → F1QualityGate')
print(f'PASS → RegisterModel → BatchTransform')
print(f'FAIL → QualityGateFailed')

Recreating pipeline with correct step dependencies...


INFO:sagemaker.processing:Uploaded pipeline_scripts to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/cx-cortalyst-sentiment-pipeline/code/47d3d5deb7601c37e0729930fb282e1f/sourcedir.tar.gz


INFO:sagemaker.processing:runproc.sh uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/cx-cortalyst-sentiment-pipeline/code/833cf1213b36bfae623195bdbf691a92/runproc.sh


INFO:sagemaker.processing:Uploaded pipeline_scripts to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/cx-cortalyst-sentiment-pipeline/code/47d3d5deb7601c37e0729930fb282e1f/sourcedir.tar.gz


INFO:sagemaker.processing:runproc.sh uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/cx-cortalyst-sentiment-pipeline/code/833cf1213b36bfae623195bdbf691a92/runproc.sh


Pipeline updated with correct dependencies
DataPreprocessing → Training → Evaluation
   → F1QualityGate
PASS → RegisterModel → BatchTransform
FAIL → QualityGateFailed


---
## 5. SUCCESS STATE — Run Pipeline with Passing F1 Threshold

F1 threshold = 0.85. Our model achieves F1 = 0.8779 → PASS → model registered.

In [29]:
execution = pipeline.start(
    execution_display_name='success-state-demo-embedding-v78',
    execution_description='CI/CD success state — F1 threshold 0.85'
)

print(f'Execution started: {execution.arn}')

Execution started: arn:aws:sagemaker:us-east-1:301798465569:pipeline/cx-cortalyst-sentiment-pipeline/execution/lulvngr7lrit


In [30]:
import time
print('Monitoring pipeline execution...')
print('(Polling every 60 seconds)\n')

while True:
    status = execution.describe()
    pipeline_status = status['PipelineExecutionStatus']
    
    # list_steps() returns a list directly
    steps = execution.list_steps()
    print(f'Pipeline status: {pipeline_status}')
    
    for step in steps:   # ← iterate directly, no .get()
        step_name   = step['StepName']
        step_status = step['StepStatus']
        icon = '✅' if step_status == 'Succeeded' else \
               '❌' if step_status == 'Failed'    else \
               '🔄' if step_status == 'Executing' else '⏳'
        print(f'  {icon} {step_name}: {step_status}')
    
    if pipeline_status in ['Succeeded', 'Failed', 'Stopped']:
        print(f'\n✅ Pipeline finished: {pipeline_status}')
        break
    
    print('---')
    time.sleep(60)

Monitoring pipeline execution...
(Polling every 60 seconds)

Pipeline status: Executing
  🔄 DataPreprocessing: Executing
---


Pipeline status: Executing
  🔄 DataPreprocessing: Executing
---


Pipeline status: Executing
  🔄 DataPreprocessing: Executing
---


Pipeline status: Executing
  🔄 DataPreprocessing: Executing
---


Pipeline status: Executing
  🔄 DataPreprocessing: Executing
---


...

Pipeline status: Executing
  🔄 BatchTransform: Executing
  ✅ RegisterModel-RegisterModel: Succeeded
  ✅ F1QualityGate: Succeeded
  ✅ CXCortalystCreateModel: Succeeded
  ✅ ModelEvaluation: Succeeded
  ✅ XGBoostTraining: Succeeded
  ✅ DataPreprocessing: Succeeded
---


Pipeline status: Executing
  🔄 BatchTransform: Executing
  ✅ RegisterModel-RegisterModel: Succeeded
  ✅ F1QualityGate: Succeeded
  ✅ CXCortalystCreateModel: Succeeded
  ✅ ModelEvaluation: Succeeded
  ✅ XGBoostTraining: Succeeded
  ✅ DataPreprocessing: Succeeded
---


Pipeline status: Executing
  🔄 BatchTransform: Executing
  ✅ RegisterModel-RegisterModel: Succeeded
  ✅ F1QualityGate: Succeeded
  ✅ CXCortalystCreateModel: Succeeded
  ✅ ModelEvaluation: Succeeded
  ✅ XGBoostTraining: Succeeded
  ✅ DataPreprocessing: Succeeded
---


Pipeline status: Executing
  🔄 BatchTransform: Executing
  ✅ RegisterModel-RegisterModel: Succeeded
  ✅ F1QualityGate: Succeeded
  ✅ CXCortalystCreateModel: Succeeded
  ✅ ModelEvaluation: Succeeded
  ✅ XGBoostTraining: Succeeded
  ✅ DataPreprocessing: Succeeded
---


Pipeline status: Succeeded
  ✅ BatchTransform: Succeeded
  ✅ RegisterModel-RegisterModel: Succeeded
  ✅ F1QualityGate: Succeeded
  ✅ CXCortalystCreateModel: Succeeded
  ✅ ModelEvaluation: Succeeded
  ✅ XGBoostTraining: Succeeded
  ✅ DataPreprocessing: Succeeded

✅ Pipeline finished: Succeeded


In [31]:
# Verify success state results
print('=== SUCCESS STATE RESULTS ===')

# Check evaluation metrics
try:
    eval_obj = s3_client.get_object(
        Bucket=BUCKET,
        Key='pipeline/evaluation/evaluation.json'
    )
    eval_report = json.loads(eval_obj['Body'].read())
    metrics = eval_report['binary_classification_metrics']
    
    f1_achieved  = metrics['f1_macro']['value']
    acc_achieved = metrics['accuracy']['value']
    auc_achieved = metrics['auc_roc']['value']
    
    print(f'\nModel Evaluation Results:')
    print(f'  Macro F1:   {f1_achieved:.4f}  (threshold: {F1_THRESHOLD}) → {"PASS" if f1_achieved >= F1_THRESHOLD else "FAIL"}')
    print(f'  Accuracy:   {acc_achieved:.4f}')
    print(f'  AUC-ROC:    {auc_achieved:.4f}')
    
except Exception as e:
    print(f'Evaluation report: {e}')

# Check model registry
packages = sm_client.list_model_packages(
    ModelPackageGroupName=MODEL_GROUP,
    SortBy='CreationTime',
    SortOrder='Descending',
    MaxResults=3
)

print(f'\nModel Registry — Latest versions:')
for pkg in packages['ModelPackageSummaryList']:
    print(f'  v{pkg["ModelPackageVersion"]} | '
          f'{pkg["ModelApprovalStatus"]} | '
          f'{pkg["CreationTime"].strftime("%Y-%m-%d %H:%M")}')

print(f'\n SUCCESS STATE DEMO COMPLETE')
print(f'   Pipeline ran end-to-end')
print(f'   Model registered and approved')
print(f'   Batch predictions written to S3')

=== SUCCESS STATE RESULTS ===

Model Evaluation Results:
  Macro F1:   0.8930  (threshold: 0.85) → PASS
  Accuracy:   0.8930
  AUC-ROC:    0.9639



Model Registry — Latest versions:
  v7 | Approved | 2026-06-22 03:04
  v6 | Approved | 2026-06-21 11:09
  v5 | Approved | 2026-06-15 16:36

 SUCCESS STATE DEMO COMPLETE
   Pipeline ran end-to-end
   Model registered and approved
   Batch predictions written to S3


---
## 5b. Map Pipeline Output → GenAI Input (Curated Parquet)

The DAG's **BatchTransform** step emits raw XGBoost scores as CSV to `pipeline/predictions/`. This step curates them into `data/predictions/production_predictions.parquet` — adding `review_id` and `predicted_label` — which is exactly the file the decoupled **GenAI module (Phase 7)** consumes. In production, the S3 write of this parquet is the event that triggers the GenAI module.

In [ ]:
# ── Map CI/CD pipeline output → GenAI input (curated parquet) ────────────────────
print('Curating pipeline predictions → parquet for the GenAI module...')

# 1. Read raw batch-transform output (CSV: one probability per line, no header)
df_pred = wr.s3.read_csv(path=S3_PREDICTIONS, header=None)
df_pred.columns = ['predicted_proba']
df_pred['predicted_label'] = (df_pred['predicted_proba'] >= 0.5).astype(int)

# 2. Restore review_id from the production split.
#    Row order matches the BatchTransform input (data/sagemaker_input/production/),
#    same alignment approach proven in 04_model.ipynb.
df_prod = wr.s3.read_parquet(path=f's3://{BUCKET}/data/splits/production/')
assert len(df_pred) == len(df_prod), (
    f'Row mismatch: {len(df_pred)} predictions vs {len(df_prod)} production rows'
)
df_pred['review_id'] = df_prod['review_id'].values

# 3. Write the curated parquet the GenAI module reads (S3-trigger target in production)
CURATED_PREDICTIONS = f's3://{BUCKET}/data/predictions/production_predictions.parquet'
wr.s3.to_parquet(df=df_pred, path=CURATED_PREDICTIONS, index=False)

print(f'Rows curated: {len(df_pred):,}')
print(f"Negative: {(df_pred['predicted_label'] == 0).sum():,} | "
      f"Positive: {(df_pred['predicted_label'] == 1).sum():,}")
print(f'Curated parquet → {CURATED_PREDICTIONS}')
print('This file is the input consumed by Phase 7 (07_GenAI_Response.ipynb).')

---
## 6. FAILURE STATE — Force Pipeline Rejection

Set F1 threshold to 0.99 — impossible to achieve → Step 4 condition FAILS → pipeline halts.

In [32]:
print('=== FAILURE STATE DEMO ===')
print('Setting F1 threshold to 0.99 (impossible to achieve)')
print('Model achieves 0.8779 → FAIL expected\n')

# Update condition with impossible threshold
FAIL_THRESHOLD = 0.99

cond_f1_fail = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_evaluate.name,
        property_file=evaluation_report,
        json_path='binary_classification_metrics.f1_macro.value'
    ),
    right=FAIL_THRESHOLD
)

step_condition_fail = ConditionStep(
    name='F1QualityGate',
    conditions=[cond_f1_fail],
    if_steps=[step_register],
    else_steps=[step_fail]
)

# Create failure pipeline
pipeline_fail = Pipeline(
    name=f'{PIPELINE_NAME}-failure-demo',
    steps=[
        step_process,
        step_train,
        step_evaluate,
        step_condition_fail
    ],
    sagemaker_session=sagemaker_session
)

pipeline_fail.upsert(role_arn=role)


INFO:sagemaker.processing:Uploaded pipeline_scripts to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/cx-cortalyst-sentiment-pipeline-failure-demo/code/47d3d5deb7601c37e0729930fb282e1f/sourcedir.tar.gz


INFO:sagemaker.processing:runproc.sh uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/cx-cortalyst-sentiment-pipeline-failure-demo/code/833cf1213b36bfae623195bdbf691a92/runproc.sh


=== FAILURE STATE DEMO ===
Setting F1 threshold to 0.99 (impossible to achieve)
Model achieves 0.8779 → FAIL expected



INFO:sagemaker.processing:Uploaded pipeline_scripts to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/cx-cortalyst-sentiment-pipeline-failure-demo/code/47d3d5deb7601c37e0729930fb282e1f/sourcedir.tar.gz


INFO:sagemaker.processing:runproc.sh uploaded to s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/cx-cortalyst-sentiment-pipeline-failure-demo/code/833cf1213b36bfae623195bdbf691a92/runproc.sh


{'PipelineArn': 'arn:aws:sagemaker:us-east-1:301798465569:pipeline/cx-cortalyst-sentiment-pipeline-failure-demo',
 'PipelineVersionId': 3,
 'ResponseMetadata': {'RequestId': '348e9939-0b7c-4d48-b880-a4b018e918e0',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '348e9939-0b7c-4d48-b880-a4b018e918e0',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '134',
   'date': 'Mon, 22 Jun 2026 03:12:58 GMT'},
  'RetryAttempts': 0}}

In [33]:
# Start failure execution
execution_fail = pipeline_fail.start(
    execution_display_name='failure-state-demo',
    execution_description=f'CI/CD FAILURE state demo — F1 threshold {FAIL_THRESHOLD}, impossible to achieve'
)

print(f'Failure pipeline started!')
print(f'   Execution ARN: {execution_fail.arn}')

Failure pipeline started!
   Execution ARN: arn:aws:sagemaker:us-east-1:301798465569:pipeline/cx-cortalyst-sentiment-pipeline-failure-demo/execution/aidbe4xz5tvg


In [34]:
import time

# Monitor failure pipeline
print('Monitoring failure pipeline execution...')

while True:
    status = execution_fail.describe()
    pipeline_status = status['PipelineExecutionStatus']
    
    # SageMaker SDK returns a list directly here
    steps = execution_fail.list_steps()
    
    print(f'Pipeline status: {pipeline_status}')
    
    # Iterate directly over the list
    for step in steps:
        step_name   = step['StepName']
        step_status = step['StepStatus']
        icon = '✅' if step_status == 'Succeeded' else \
               '❌' if step_status == 'Failed' else \
               '🔄' if step_status == 'Executing' else '⏳'
        print(f'  {icon} {step_name}: {step_status}')
    
    if pipeline_status in ['Succeeded', 'Failed', 'Stopped']:
        print(f'\n📌 Failure pipeline finished: {pipeline_status}')
        if pipeline_status == 'Failed':
            print(f'   ✅ Quality gate correctly rejected the model!')
            print(f'   ✅ Production model preserved unchanged')
        break
    
    print('---')
    time.sleep(60)

Monitoring failure pipeline execution...
Pipeline status: Executing
  🔄 DataPreprocessing: Executing
---


Pipeline status: Executing
  🔄 DataPreprocessing: Executing
---


Pipeline status: Executing
  🔄 DataPreprocessing: Executing
---


Pipeline status: Executing
  🔄 DataPreprocessing: Executing
---


Pipeline status: Executing
  🔄 DataPreprocessing: Executing
---


...

Pipeline status: Executing
  🔄 XGBoostTraining: Executing
  ✅ DataPreprocessing: Succeeded
---


Pipeline status: Executing
  🔄 XGBoostTraining: Executing
  ✅ DataPreprocessing: Succeeded
---


Pipeline status: Executing
  🔄 ModelEvaluation: Executing
  ✅ XGBoostTraining: Succeeded
  ✅ DataPreprocessing: Succeeded
---


Pipeline status: Executing
  🔄 ModelEvaluation: Executing
  ✅ XGBoostTraining: Succeeded
  ✅ DataPreprocessing: Succeeded
---


Pipeline status: Failed
  ❌ QualityGateFailed: Failed
  ✅ F1QualityGate: Succeeded
  ✅ ModelEvaluation: Succeeded
  ✅ XGBoostTraining: Succeeded
  ✅ DataPreprocessing: Succeeded

📌 Failure pipeline finished: Failed
   ✅ Quality gate correctly rejected the model!
   ✅ Production model preserved unchanged


---
## 7. Pipeline Summary & Video Demo Guide

In [35]:
print(f'''
╔══════════════════════════════════════════════════════════════╗
║      CX CORTALYST — CI/CD PIPELINE SUMMARY                   ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  PIPELINE NAME: {PIPELINE_NAME:<44} ║
║                                                              ║
║  STEPS                                                       ║
║  Step 1: DataPreprocessing                                   ║
║  Step 2: XGBoostTraining    (ml.m5.xlarge, 100 rounds)       ║
║  Step 3: ModelEvaluation    (F1, Accuracy, AUC-ROC)          ║
║  Step 4: F1QualityGate      (threshold: 0.85)                ║
║    PASS  → RegisterModel + BatchTransform                    ║
║    FAIL  → QualityGateFailed (halts)                         ║
║                                                              ║
║  SUCCESS STATE                                               ║
║  F1 threshold: 0.85 | Model achieves: 0.8910 → PASS          ║
║  Model registered in cx-cortalyst-sentiment-models           ║
║  Predictions written to s3://.../pipeline/predictions/       ║
║                                                              ║
║  FAILURE STATE                                               ║
║  F1 threshold: 0.99 | Model achieves: 0.8910 → FAIL          ║
║  Pipeline halts at QualityGateFailed step                    ║
║  Production model preserved unchanged                        ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
''')


╔══════════════════════════════════════════════════════════════╗
║      CX CORTALYST — CI/CD PIPELINE SUMMARY                   ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  PIPELINE NAME: cx-cortalyst-sentiment-pipeline              ║
║                                                              ║
║  STEPS                                                       ║
║  Step 1: DataPreprocessing                                   ║
║  Step 2: XGBoostTraining    (ml.m5.xlarge, 100 rounds)       ║
║  Step 3: ModelEvaluation    (F1, Accuracy, AUC-ROC)          ║
║  Step 4: F1QualityGate      (threshold: 0.85)                ║
║    PASS  → RegisterModel + BatchTransform                    ║
║    FAIL  → QualityGateFailed (halts)                         ║
║                                                              ║
║  SUCCESS STATE                                               ║
║  F1 threshold: 0.85 | 